# Reproducing Mountoufaris et al. (2024) from dandiset 001131: rSLDS fits, seeds 0 and 1

Paper. Mountoufaris G, Nair A, et al. (2024), *Cell* 187, 5998-6015,
[doi:10.1016/j.cell.2024.08.015](https://doi.org/10.1016/j.cell.2024.08.015); the claims file was
written from the accepted author manuscript (PMC11490375). Dandiset
[DANDI:001131](https://dandiarchive.org/dandiset/001131/0.260601.1203), version 0.260601.1203, the
most recent published version (it holds the two files of the earlier version, 0.240826.1647, and
eight more; see `reproduce_mountoufaris_2024.ipynb`, which also holds the data checks).

The claims file asks for fits with seeds 0 to 3, each claim's outcome being the one all seeds give.
One fit of all ten mice takes about 15 min on four cores, so the seeds are split over two notebooks,
each under the 60-min limit: this one fits seeds 0 and 1. The outcome of each claim combines both
notebooks and is given in the attempt record; the conclusions here are for these seeds.

Settings (claims file, fig6-rslds-fit-quality, from the ssm notebook the paper's code points to):
one model per mouse, 3 discrete states, 5 latent dimensions, recurrent_only transitions,
diagonal_gaussian dynamics, gaussian_orthog emissions, single subspace, Laplace-EM with a structured
mean-field posterior, 100 iterations, 0.1 s bins. All ten mice have at least 31 units. The files hold
only the male-intruder part of each session, so the two readings of which frames are fit (the
male-intruder part, or the whole session) are the same: the whole file. The files hold z-scored
traces; the sigma-normalized reading divides each unit by a baseline SD and is not fit (see the
deviations in the attempt record).

| Claim | Here |
|---|---|
| fig6-rslds-fit-quality | tested |
| fig6a-control-tau1-greater | tested |
| fig6c-experimental-tau-indistinguishable | tested |
| fig6f-line-attractor-score | tested |
| fig6g-flow-field-line-vs-point | tested |
| fig7b-achw-first-dimension | tested |
| fig6e-integration-dimension-persistence | blocked, metadata insufficient; a variant is shown |
| figs6f-mating-attractor-intact | blocked, data missing (no female-intruder data) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vmh

print("ssm commit", vmh.ensure_ssm())
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

SEEDS = (0, 1)
started = time.time()
mice = vmh.load_all()
G = {m["mouse"]: m["group"] for m in mice}
M = {m["mouse"]: m for m in mice}
print(f"loaded {len(mice)} files from dandiset {vmh.DANDISET_ID} version {vmh.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
print(pd.DataFrame([{"mouse": m["mouse"], "group": m["group"], "frames": m["Y"].shape[0],
                     "units": m["Y"].shape[1], "at_least_31_units": m["Y"].shape[1] >= 31,
                     "attack_bouts": len(m["bouts"].get("attack", [])),
                     "sniff_bouts": len(m["bouts"]["sniff"]),
                     "trace_min": m["Y"].min(), "trace_max": m["Y"].max()} for m in mice]).round(2).to_string(index=False))
print("seeds:", SEEDS, "; states", vmh.K_STATES, "; latent dimensions", vmh.D_LATENT, "; iterations", vmh.NUM_ITERS)

## Fits

In [ ]:
t0 = time.time()
fits = vmh.fit_all(mice, seeds=SEEDS)
print(f"{len(fits)} fits in {(time.time() - t0) / 60:.1f} min")
S = {key: vmh.summarize_fit(f, M[key[0]]) for key, f in fits.items()}
rows = []
for (mouse, seed), s in sorted(S.items(), key=lambda kv: (G[kv[0][0]], kv[0])):
    f = fits[(mouse, seed)]
    rows.append({"mouse": mouse, "group": G[mouse], "seed": seed,
                 "elbo_first": f["elbos"][0], "elbo_last": f["elbos"][-1],
                 "elbo_change_last10_pct": 100 * (f["elbos"][-1] - f["elbos"][-11]) / abs(f["elbos"][-1]),
                 "r2": s["r2"], "fwd_r2_1s": vmh.forward_sim_score(f, 1.0), "fwd_r2_5s": vmh.forward_sim_score(f, 5.0),
                 "state_from": s["state_from"], "state": s["attack_state"],
                 "attack_in_state": s["attack_frac_in_state"],
                 **{f"occ{k}": s["state_occupancy"][k] for k in range(vmh.K_STATES)},
                 **{f"tau{j + 1}_a": s["tau_a"][j] for j in range(vmh.D_LATENT)},
                 **{f"tau{j + 1}_b": s["tau_b"][j] for j in range(vmh.D_LATENT)},
                 "score_a": s["score_a"], "score_b": s["score_b"]})
fit_table = pd.DataFrame(rows)
print(fit_table.round(3).to_string(index=False))

`tau*_a` are the time constants (s) of the state holding most attack frames (reading (a); in mice 510
and 500, which have no attack annotation, the state holding most sniffing frames, `state_from`);
`tau*_b` the mean over the three states (reading (b)). The ELBO's change over the last 10 iterations
shows how far each fit had converged. A pair of complex eigenvalues gives two equal time constants,
and so a line attractor score of 0 under the Methods' formula.

### Link to the authors' fits

Mice 510 and 500 carry the authors' weights of each unit on "latent factor 1". If these fits find
the same integration dimension, the weights of their first dimension should correlate with them.

In [ ]:
rows = []
for (mouse, seed), s in S.items():
    w = M[mouse]["rslds_weights"]
    if w is None:
        continue
    r = {"mouse": mouse, "seed": seed}
    for j in range(vmh.D_LATENT):
        r[f"dim{j + 1}"] = abs(stats.spearmanr(np.abs(w), s["weights"][:, j])[0])
    rows.append(r)
link = pd.DataFrame(rows)
print("|Spearman r| between the published |weights| and each fit dimension's |C v_j|:")
print(link.round(3).to_string(index=False))
print("published weights: range %.3f to %.3f (510), %.3f to %.3f (500)" % (
    M["ctrlm510"]["rslds_weights"].min(), M["ctrlm510"]["rslds_weights"].max(),
    M["mutm500"]["rslds_weights"].min(), M["mutm500"]["rslds_weights"].max()))

## Claim fig6-rslds-fit-quality

> rSLDS models with 5 latent dimensions and 3 discrete states, fit to each mouse with at least 31
> units, capture about 85% of the variance, with similar performance in control and experimental
> mice; attack occurs within a single rSLDS state in both groups.

Variance explained is the R2 of the emissions reconstruction (claims file). Performance is the R2 of
forward predictions of the latents over 1 s and 5 s; the paper computes it on held-out folds, which
would take five more fits per mouse and seed and is not run here, so it is computed on the fit data.

In [ ]:
rows = []
for seed in SEEDS:
    d = fit_table[fit_table.seed == seed]
    c, e = d[d.group == "control"], d[d.group == "experimental"]
    att = d[d.state_from == "attack"]
    rows.append({"seed": seed, "r2_control": c.r2.mean(), "r2_exp": e.r2.mean(),
                 "r2_min": d.r2.min(), "r2_max": d.r2.max(),
                 "fwd1_control": c.fwd_r2_1s.mean(), "fwd1_exp": e.fwd_r2_1s.mean(),
                 "p_fwd1": vmh.mw(c.fwd_r2_1s, e.fwd_r2_1s)[1],
                 "fwd5_control": c.fwd_r2_5s.mean(), "fwd5_exp": e.fwd_r2_5s.mean(),
                 "p_fwd5": vmh.mw(c.fwd_r2_5s, e.fwd_r2_5s)[1],
                 "mice_with_attack": len(att), "attack_majority_one_state": int((att.attack_in_state > 0.5).sum()),
                 "attack_90pct_one_state": int((att.attack_in_state >= 0.9).sum()),
                 "attack_in_state_min": att.attack_in_state.min()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("expected variance explained 85 +/- 8.5 %")

**Result: not reproduced for seeds 0 and 1.** The fits explain 59% of the variance in both groups
(0.594 control, 0.586 to 0.587 experimental; range 0.496 to 0.664 over mice), against 85 +/- 8.5%.
The forward-prediction R2 does not differ between groups (Mann-Whitney p = 0.114 at 1 s and 5 s,
both seeds), as the paper reports, though it is computed on the fit data and not on held-out folds.
Attack frames fall mostly in one state in 6 (seed 0) and 7 (seed 1) of the 8 mice with attack
annotations, and at least 90% in one state in 4 and 3 of them. The link check above finds that no
fit dimension of mice 510 and 500 matches the authors' latent factor 1 well (|Spearman r| at most
0.46).

## Claim fig6a-control-tau1-greater

> In control mice, the time constant of the first rSLDS dimension (about 100-120 s) is
> significantly larger than that of the second (about 40 s).

In [ ]:
def tau_tests(group):
    rows = []
    for seed in SEEDS:
        d = fit_table[(fit_table.seed == seed) & (fit_table.group == group)]
        for reading in ("a", "b"):
            T = d[[f"tau{j + 1}_{reading}" for j in range(vmh.D_LATENT)]].to_numpy()
            kw = stats.kruskal(*T.T)
            rows.append({"seed": seed, "reading": reading, "mice": len(T),
                         **{f"mean_tau{j + 1}": T[:, j].mean() for j in range(vmh.D_LATENT)},
                         "kw_H": kw.statistic, "kw_p": kw.pvalue, "dunn_1v2_p": vmh.dunn_1_vs_2(T),
                         "tau1_largest_mean": bool(T[:, 0].mean() == T.mean(0).max())})
    return pd.DataFrame(rows)

ctl_tau = tau_tests("control")
print(ctl_tau.round(3).to_string(index=False))
print("expected control tau1 90 to 132 s, tau2 36 to 44 s")

**Result: not reproduced for seeds 0 and 1.** Control time constants are far longer than the
paper's: mean tau1 279 to 665 s and tau2 274 to 488 s over seeds and readings, against 90 to 132 s
and 36 to 44 s. Kruskal-Wallis across the five sorted dimensions gives p = 0.003 and 0.002 for
reading (b) and 0.050 and 0.057 for reading (a), and Dunn's comparison of dimension 1 with 2 gives p
= 1.0 in every case. In reading (a) tau1 and tau2 are often equal, because the slowest eigenvalues
of the state's matrix are a complex pair.

## Claim fig6c-experimental-tau-indistinguishable

> In experimental mice the first two rSLDS dimensions have statistically indistinguishable time
> constants (below 50 s), because the first dimension's time constant is reduced relative to controls.

In [ ]:
exp_tau = tau_tests("experimental")
print(exp_tau.round(3).to_string(index=False))
rows = []
for seed in SEEDS:
    d = fit_table[fit_table.seed == seed]
    for reading in ("a", "b"):
        c, e = d[d.group == "control"][f"tau1_{reading}"], d[d.group == "experimental"][f"tau1_{reading}"]
        rows.append({"seed": seed, "reading": reading, "tau1_control": c.mean(), "tau1_exp": e.mean(),
                     "p_mw_tau1": vmh.mw(c, e)[1], "p_mw_tau1_control_greater": vmh.mw(c, e, "greater")[1]})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("smallest attainable two-sided p with 4 and 6 mice:", round(2 / 210, 4))

**Result: not reproduced for seeds 0 and 1.** Experimental mean tau1 is 137 to 825 s and tau2 118 to
783 s, not below 55 s. The paper's Kruskal-Wallis test over the five dimensions finds a difference
(p from 0.001 to 0.006), while Dunn's comparison of dimension 1 with 2 does not (p = 1.0). Control
tau1 is not longer than experimental tau1 at p < 0.01 under any seed or reading (two-sided p =
0.038, 0.17, 0.91 and 0.067), and in two of the four cases the experimental mean is the longer.

## Claim fig6f-line-attractor-score

> The line attractor score, log2 of the ratio of the two largest time constants, is about 1.6 in
> control mice and close to zero in experimental mice.

In [ ]:
rows = []
for seed in SEEDS:
    d = fit_table[fit_table.seed == seed]
    for reading in ("a", "b"):
        c, e = d[d.group == "control"][f"score_{reading}"], d[d.group == "experimental"][f"score_{reading}"]
        rows.append({"seed": seed, "reading": reading, "score_control": c.mean(), "score_exp": e.mean(),
                     "control_values": np.round(c.to_numpy(), 2), "exp_values": np.round(e.to_numpy(), 2),
                     "p_mw": vmh.mw(c, e)[1], "p_mw_control_greater": vmh.mw(c, e, "greater")[1]})
scores = pd.DataFrame(rows)
print(scores.round(4).to_string(index=False))
print("expected control mean 1.44 to 1.76; experimental mean below 1")

**Result: not reproduced for seeds 0 and 1.** The mean control score is 0.00 to 0.68 (expected 1.44
to 1.76) and the experimental mean 0.07 to 0.46; the Mann-Whitney test finds no difference (p from
0.15 to 1.0). Most scores under reading (a) are 0, as the two slowest eigenvalues are a complex
pair.

## Claim fig6g-flow-field-line-vs-point

> Flow fields of the fit rSLDS in the plane of the first two principal components show a line
> attractor (a roughly linear region of low flow) in control mice and a point attractor in
> experimental mice.

Judged per mouse by the line attractor score (claims file). The flow field of the state used for
reading (a) is drawn for the two mice whose files the authors chose as exemplars (510 and 500), for
the first seed of this notebook.

In [ ]:
rows = []
for seed in SEEDS:
    d = fit_table[fit_table.seed == seed]
    for reading in ("a", "b"):
        c, e = d[d.group == "control"][f"score_{reading}"], d[d.group == "experimental"][f"score_{reading}"]
        rows.append({"seed": seed, "reading": reading, "control_above_1": f"{int((c > 1).sum())} of {len(c)}",
                     "exp_below_1": f"{int((e < 1).sum())} of {len(e)}",
                     "reproduced": bool((c > 1).sum() > len(c) / 2 and (e < 1).sum() > len(e) / 2)})
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, mouse in zip(axes, ("ctrlm510", "mutm500")):
    f, s = fits[(mouse, SEEDS[0])], S[(mouse, SEEDS[0])]
    x = f["x"]
    mu = x.mean(0)
    _, _, Vt = np.linalg.svd(x - mu, full_matrices=False)
    P = Vt[:2].T                       # latent -> first two PCs
    pc = (x - mu) @ P
    g1, g2 = np.meshgrid(np.linspace(*np.percentile(pc[:, 0], [1, 99]), 25),
                         np.linspace(*np.percentile(pc[:, 1], [1, 99]), 25))
    grid = mu + np.column_stack([g1.ravel(), g2.ravel()]) @ P.T
    A, b = f["As"][s["attack_state"]], f["bs"][s["attack_state"]]
    dx = (grid @ A.T + b - grid) @ P
    speed = np.linalg.norm(dx, axis=1)
    ax.contourf(g1, g2, np.log10(speed.reshape(g1.shape)), 20, cmap="Greys_r")
    ax.quiver(g1, g2, dx[:, 0].reshape(g1.shape), dx[:, 1].reshape(g1.shape), color="tab:blue")
    ax.plot(pc[:, 0], pc[:, 1], "r", lw=0.3, alpha=0.5)
    ax.set(title=f"{mouse}, seed {SEEDS[0]}, state {s['attack_state']}", xlabel="PC1", ylabel="PC2")
    inside = np.mean((pc[:, 0] >= g1.min()) & (pc[:, 0] <= g1.max()) & (pc[:, 1] >= g2.min()) & (pc[:, 1] <= g2.max()))
    print(f"{mouse}: speed range {speed.min():.4f} to {speed.max():.4f} per bin; "
          f"{100 * (1 - inside):.1f}% of trajectory points outside the axes; score_a {s['score_a']:.2f}")
plt.tight_layout(); plt.show()

**Result: not reproduced for seeds 0 and 1.** At most 2 of the 4 control mice have a score above 1
under any seed and reading (0, 2, 0 and 1 of 4), so a majority never does; 5 or 6 of the 6
experimental mice have a score below 1. In the flow fields drawn above, both mouse 510 (control) and
mouse 500 (experimental) show a single low-flow point, with flow rotating around it in mouse 510,
and no line-shaped low-flow region.

## Claim fig7b-achw-first-dimension

> The mean autocorrelation half-width of units weighted on the first rSLDS dimension, over the entire
> male-male interaction, is shorter in experimental (9.86 +/- 1 s) than in control mice (28.6 +/- 1.23
> s); the reduction is about 66%, against about 61% for second-dimension units.

ACHW per unit over the whole interaction (first annotated behaviour to the end of the file). ACHW is
the same for z-scored and sigma-normalized traces (an affine map does not change the
autocorrelation), and with one intruder per file the "mean over trials" reading is the single
autocorrelation, so the readings left are which units are "weighted on" a dimension: (a) the
dimension of each unit's largest weight, (b) the top 25% of units by weight on that dimension.

In [ ]:
achw_all = {m["mouse"]: vmh.achw(m["Y"][vmh.window_frames(m, 1e6)]) for m in mice}
rows = []
for seed in SEEDS:
    for reading in ("largest", "top25"):
        v1, v2 = {}, {}
        for m in mice:
            W = S[(m["mouse"], seed)]["weights"]
            if reading == "largest":
                best = W.argmax(1)
                s1, s2 = best == 0, best == 1
            else:
                s1 = W[:, 0] >= np.quantile(W[:, 0], 0.75)
                s2 = W[:, 1] >= np.quantile(W[:, 1], 0.75)
            v1[m["mouse"]], v2[m["mouse"]] = achw_all[m["mouse"]][s1], achw_all[m["mouse"]][s2]
        r1, r2 = vmh.nested_tests(v1, G), vmh.nested_tests(v2, G)
        rows.append({"seed": seed, "units": reading,
                     "dim1_units_c": r1["n_units_c"], "dim1_units_e": r1["n_units_e"],
                     "dim1_control": r1["pooled_mean_c"], "dim1_exp": r1["pooled_mean_e"],
                     "dim1_control_mice": r1["mouse_mean_c"], "dim1_exp_mice": r1["mouse_mean_e"],
                     "dim1_p_mw_pooled": r1["p_mw_pooled"], "dim1_p_mw_mice": r1["p_mw_mice"],
                     "dim2_reduction_pct": 100 * (1 - r2["pooled_mean_e"] / r2["pooled_mean_c"]),
                     "dim2_reduction_mice_pct": 100 * (1 - r2["mouse_mean_e"] / r2["mouse_mean_c"]),
                     "dim2_p_ks_pooled": r2["p_ks_pooled"], "dim2_p_mw_mouse_medians": r2["p_mw_mouse_medians"]})
ach1 = pd.DataFrame(rows)
print(ach1.round(4).to_string(index=False))
print("expected: dim-1 experimental 9.86 +/- 1 s, control 28.6 +/- 1.23 s; dim-2 reduction 61 +/- 6.1 %")
x = np.concatenate(list(achw_all.values()))
print(f"ACHW over the whole interaction, all units: range {np.nanmin(x):.1f} to {np.nanmax(x):.1f} s, "
      f"{int(np.isnan(x).sum())} undefined")

**Result: not reproduced for seeds 0 and 1.** The ACHW of first-dimension units is shorter in
experimental mice under every combination (pooled means 17.4 to 20.8 s against 26.7 to 31.4 s;
pooled Mann-Whitney p < 0.001; on per-mouse means p = 0.17 to 0.61). The experimental values are
about twice the paper's 9.86 +/- 1 s in every combination, and no control value falls inside 28.6
+/- 1.23 s (26.7, 29.9, 30.3 and 31.4 s). The second-dimension reduction is 36 to 78% where it can
be computed, against 61 +/- 6.1%. When the two slowest eigenvalues are a complex pair, the real
parts of their eigenvectors are the same, so dimensions 1 and 2 get the same weights: under the
largest-weight reading no unit is then assigned to dimension 2 (seed 0 has none at all, NaN), and
under the top-25% reading the same units are taken for both.

## Blocked claim fig6e-integration-dimension-persistence, a variant

The files do not mark when the intruder is removed, so the post-intruder period of Figure 6E cannot
be found. As a variant that does not decide the outcome, the first dimension's activity (projection
of the latents on the first eigenvector of the reading (a) state, normalized to 0-1 per mouse) is
compared between all frames of annotated behaviour and the frames after the last annotated behaviour.

In [ ]:
rows = []
for (mouse, seed), s in S.items():
    m, f = M[mouse], fits[(mouse, seed)]
    A = f["As"][s["attack_state"]]
    tau, V = vmh.time_constants(A)
    p = f["x"] @ np.real(V[:, 0])
    p = (p - p.min()) / (p.max() - p.min())
    n = len(p)
    beh = np.zeros(n, bool)
    for b in m["bouts"].values():
        beh |= vmh.frames(b, n)
    last = int(max(b[-1, 1] for b in m["bouts"].values()) * m["rate"])
    first = int(vmh.first_behaviour_s(m) * m["rate"])
    if first > 0 and p[beh].mean() < p[:first].mean():
        p = 1 - p                      # orient the axis so that activity during bouts is above the pre-interaction level
    on_bouts, after = p[beh].mean(), p[last:].mean()
    rows.append({"mouse": mouse, "group": m["group"], "seed": seed, "after_s": (n - last) / m["rate"],
                 "during_bouts": on_bouts, "after_last_bout": after})
post = pd.DataFrame(rows)
print(post.round(3).to_string(index=False))
for seed in SEEDS:
    for g in ("control", "experimental"):
        d = post[(post.seed == seed) & (post.group == g)]
        t = stats.ttest_rel(d.during_bouts, d.after_last_bout)
        print(f"seed {seed} {g}: during {d.during_bouts.mean():.3f}, after {d.after_last_bout.mean():.3f}, "
              f"paired t = {t.statistic:.2f}, p = {t.pvalue:.3f}")

The sign of an eigenvector is arbitrary, so each mouse's axis is oriented so that activity during
bouts is above the level before the first annotated behaviour; the variant shows whether activity on the slowest dimension differs after the last
annotated behaviour, not whether it stays elevated after an intruder's removal.

## Summary

| Claim | Seeds 0 and 1 |
|---|---|
| fig6-rslds-fit-quality | not reproduced: variance explained 59%, not 85% |
| fig6a-control-tau1-greater | not reproduced: control tau1 279 to 665 s; tau1 and tau2 not separated (Dunn p = 1.0) |
| fig6c-experimental-tau-indistinguishable | not reproduced: experimental tau1 and tau2 well above 55 s; tau1 not lower than control at p < 0.01 |
| fig6f-line-attractor-score | not reproduced: control score 0.00 to 0.68 |
| fig6g-flow-field-line-vs-point | not reproduced: at most 2 of 4 control mice above 1 |
| fig7b-achw-first-dimension | not reproduced: experimental ACHW 17 to 21 s, not 9.9 s |
| fig6e-integration-dimension-persistence | blocked, metadata insufficient |
| figs6f-mating-attractor-intact | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: the fits use the z-scored traces of the whole file (the
files hold only the male-intruder part), the performance score is computed on the fit data and not on
held-out folds, and mice 510 and 500 have no attack annotation, so their reading (a) state is the one
holding most sniffing frames. The fits do not recover the authors' latent factor 1 deposited for mice
510 and 500, so the fits here and the paper's may differ in ways the Methods do not let one find.